In [1]:
print('hello world')

hello world


In [1]:
import pandas as pd

In [5]:
df = pd.read_excel('Данные ЗУП.xlsx', skiprows=2, header=None)

In [7]:
# Назначаем имена колонок (согласно структуре вашего файла)
# 0 - Наименование (Сотрудник/Начисление), 1 - Подразделение, 2 - Должность
# 4 - Плановый ФОТ, 6 - Начислено
col_names = {
    0: 'raw_name', 
    1: 'department', 
    2: 'position', 
    4: 'planned_fot', 
    6: 'actual_amount'
}

In [8]:
df = df.rename(columns=col_names)

In [9]:
df

,raw_name,department,position,3,planned_fot,5,actual_amount,7,8,9,10,11,12,13
0,Желяк Егор Андреевич,Основное подразделение,Программист интерфейсов,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Оплата по окладу,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Зацаринный Станислав Сергеевич,Основное подразделение,Системный инженер,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Оплата по окладу,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Подолякина Анна Михайловна,Основное подразделение,Заместитель главного бухгалтера,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366,Оплата по окладу,NaN,NaN,NaN,317000.0,NaN,317000.00,NaN,NaN,NaN,22.0,176.0,22.0,176.0
367,Премия по итогам работы за 2025год,NaN,NaN,NaN,NaN,NaN,317000.00,-317000.00,-100.0,NaN,NaN,NaN,NaN,NaN
368,Черновол Никита Владимирович,"ОП Краснодар Березанская ООО ""Винтео""",Специалист по автоматизации тестирования,NaN,136850.0,NaN,134242.33,2607.67,1.9,NaN,22.0,176.0,12.0,96.0
369,Оплата по окладу,NaN,NaN,NaN,136850.0,NaN,74645.45,62204.55,45.5,NaN,22.0,176.0,12.0,96.0


In [10]:
# 2. Идентификация строк сотрудников
# В отчетах ЗУП у строки сотрудника заполнены подразделение и должность, 
# а у строк начислений эти поля обычно пустые.
is_employee_row = df['department'].notna()

In [11]:
is_employee_row

0       True
1      False
2       True
3      False
4       True
       ...  
366    False
367    False
368     True
369    False
370    False
Name: department, Length: 371, dtype: bool

In [12]:
# 3. Создаем "плоские" колонки
# Копируем имя только если это строка сотрудника
df['Сотрудник'] = df['raw_name'].where(is_employee_row)
df['Подразделение_фикс'] = df['department']
df['Должность_фикс'] = df['position']

In [ ]:
# 4. Магия ffill: "протягиваем" данные сотрудника вниз на его начисления
df[['Сотрудник', 'Подразделение_фикс', 'Должность_фикс']] = df[['Сотрудник', 'Подразделение_фикс', 'Должность_фикс']].ffill()

In [16]:
# 5. Определяем вид начисления
# В колонке raw_name теперь вид начисления, если это НЕ строка сотрудника
df['Вид_начисления'] = df['raw_name'].where(~is_employee_row)

In [20]:
# 6. Фильтруем результат
# Оставляем только строки с детальными начислениями (где изначально не было подразделения)
# И убираем пустые строки, если они есть
df_flat = df[~is_employee_row & df['Вид_начисления'].notna()].copy()

In [22]:
df_flat.head()

,raw_name,department,position,3,planned_fot,5,actual_amount,7,8,9,10,11,12,13,Сотрудник,Подразделение_фикс,Должность_фикс,Вид_начисления
1,Оплата по окладу,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Желяк Егор Андреевич,Основное подразделение,Программист интерфейсов,Оплата по окладу
3,Оплата по окладу,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Зацаринный Станислав Сергеевич,Основное подразделение,Системный инженер,Оплата по окладу
5,Оплата по окладу,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Подолякина Анна Михайловна,Основное подразделение,Заместитель главного бухгалтера,Оплата по окладу
7,Оплата по окладу,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Набиева Алина Александровна,Отдел внедрения,Технический писатель,Оплата по окладу
9,Оплата по окладу,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Немцев Руслан Юрьевич,Отдел внедрения,Продакт-менеджер,Оплата по окладу


In [31]:
df_res = df_flat[['Сотрудник', 'Подразделение_фикс', 'Должность_фикс', 'Вид_начисления', 'actual_amount']].copy()

In [32]:
df_res.rename(columns={'Подразделение_фикс':'Подразделение', 'Должность_фикс': "Должность", 'actual_amount':"Начислено"}, inplace=True)    

In [33]:
df_res.dropna(inplace=True)

In [37]:
df_res

,Сотрудник,Подразделение,Должность,Вид_начисления,Начислено
17,Кова Евгений Романович,Финансовый отдел,Юрист,Оплата по окладу,88863.64
18,Кова Евгений Романович,Финансовый отдел,Юрист,Премия по итогам работы за 2025год,115000.00
20,Костюк Анастасия Владиславовна,Финансовый отдел,Юрисконсульт,Оплата по окладу,126500.00
21,Костюк Анастасия Владиславовна,Финансовый отдел,Юрисконсульт,Премия по итогам работы за 2025год,63250.00
23,Попов Николай Андреевич,Финансовый отдел,Финансовый директор,Оплата по окладу,18636.36
...,...,...,...,...,...
360,Сейлер Константин Сергеевич,"ОП Краснодар Березанская ООО ""Винтео""",Инженер по автоматизации тестирования,Отпуск основной,27100.20
366,Чеботарев Алексей Евгеньевич,"ОП Краснодар Березанская ООО ""Винтео""",Старший веб-программист,Оплата по окладу,317000.00
367,Чеботарев Алексей Евгеньевич,"ОП Краснодар Березанская ООО ""Винтео""",Старший веб-программист,Премия по итогам работы за 2025год,317000.00
369,Черновол Никита Владимирович,"ОП Краснодар Березанская ООО ""Винтео""",Специалист по автоматизации тестирования,Оплата по окладу,74645.45


In [40]:
def format_fio(fullname):
    if pd.isna(fullname) or not str(fullname).strip():
        return fullname
    
    parts = str(fullname).split()
    
    if len(parts) == 1:
        return parts[0]
    
    # Берем фамилию и добавляем инициалы
    surname = parts[0]
    initials = [f"{p[0]}." for p in parts[1:3]] # Берем только имя и отчество (индексы 1 и 2)
    
    return f"{surname} {' '.join(initials)}"

In [41]:
# Применяем функцию к колонке
df_res['Сотрудник сокр'] = df_res['Сотрудник'].apply(format_fio)

In [42]:
df_pivot = pd.pivot_table(df_res, index=['Сотрудник сокр','Вид_начисления'], values='Начислено', aggfunc='sum')

In [10]:
df = pd.read_excel('Данные Т51.xlsx', skiprows=2) 

In [3]:
col_names = {
   'Unnamed: 2':"Сотрудник сокр",
   'Unnamed: 3':"Должность"   
}

In [4]:
drop_col = ["Unnamed: 0", 
            "Unnamed: 1", 
            "всего",
            "Unnamed: 4",
            "Unnamed: 5",
            "Unnamed: 7",
            "Unnamed: 8",
            "рабочих",
            "Unnamed: 24",
            "Unnamed: 25",
            "Unnamed: 26",
            "Unnamed: 27",
            "Unnamed: 28",
            "задолженности",
            "Unnamed: 30",
            "\nк\nвыплате",
            "выход-\nных и празд-\nничных",
            "Unnamed: 10",	
            "Unnamed: 11"
            ]

In [11]:
df.rename(columns=col_names, inplace=True)
df = df.dropna(subset=['Сотрудник сокр'])
df = df[df['Сотрудник сокр'] != "3"] 

In [14]:
def find_stop_column(columns):
    for i, col in enumerate(columns):
        # Превращаем заголовок в одну строку (даже если это кортеж)
        col_name = str(col).lower()
        if "всего" in col_name:
            return i
    return None

stop_idx = find_stop_column(df.columns)
if stop_idx is not None:
    df = df.iloc[:, :stop_idx]

In [16]:
df.columns

Index(['Unnamed: 0', 'Unnamed: 1', 'Сотрудник сокр', 'Должность', 'Unnamed: 4',
       'Unnamed: 5', 'рабочих', 'Unnamed: 7', 'Unnamed: 8',
       'выход-\nных и празд-\nничных', 'Unnamed: 10', 'Unnamed: 11',
       'Больничный за счет работодателя', 'Командировка',
       'Компенсация не использованных дней отдыха',
       'Компенсация отпуска (Отпуск основной)', 'Оплата по окладу',
       'Оплата работы в праздничные и выходные дни',
       'Оплата сверхурочных часов', 'Отпуск основной', 'Районный коэффициент',
       'Северная надбавка', 'Суточные сверх нормы'],
      dtype='str')

In [13]:
# 1. Превращаем названия колонок в строки и приводим к нижнему регистру
# 2. Ищем, где встречается "всего"
mask = df.columns.astype(str).str.lower().str.contains("всего")

if mask.any():
    # Находим индекс первой колонки, где маска True
    stop_col_idx = mask.values.argmax() 
    # Обрезаем таблицу
    df = df.iloc[:, :stop_col_idx]  

AttributeError: 'numpy.ndarray' object has no attribute 'values'

In [7]:
df

,Unnamed: 0,Unnamed: 1,Сотрудник сокр,Должность,Unnamed: 4,Unnamed: 5,рабочих,Unnamed: 7,Unnamed: 8,выход-\nных и празд-\nничных,...,Суточные сверх нормы,всего,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,задолженности,Unnamed: 30,\nк\nвыплате
2,1.0,NaN,Авдеев В. Е.,Младший тестировщик,119000.0,NaN,17.0,NaN,NaN,2.0,...,NaN,121704.55,77647,NaN,NaN,77647,NaN,NaN,NaN,44057.55
3,2.0,NaN,Акимов А. А.,Специалист по тестировани,69000.0,NaN,22.0,NaN,NaN,NaN,...,NaN,69000.00,30015,NaN,NaN,30015,NaN,NaN,NaN,38985.00
4,3.0,NaN,Алексеев Н. С.,Инженер по защите информа,288000.0,NaN,22.0,NaN,NaN,3.0,...,NaN,366545.45,166908.63,NaN,NaN,166908.63,NaN,NaN,NaN,199636.82
5,4.0,NaN,Барышев М. В.,Главный инженер,460000.0,NaN,22.0,NaN,NaN,NaN,...,NaN,460000.00,195500,NaN,NaN,195500,NaN,NaN,NaN,264500.00
6,5.0,NaN,Беликова Е. К.,Ведущий эксперт Винтео,126500.0,NaN,12.0,NaN,NaN,NaN,...,NaN,69000.00,55028,NaN,NaN,55028,NaN,NaN,NaN,13972.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,87.0,NaN,Чмель В. В.,Тестировщик-стажер,57500.0,NaN,17.0,NaN,NaN,NaN,...,NaN,44431.82,25013,NaN,NaN,25013,NaN,NaN,NaN,19418.82
89,88.0,NaN,Чумаков К. В.,Инженер платформенной ком,190500.0,NaN,22.0,NaN,NaN,NaN,...,NaN,190500.00,82867,NaN,NaN,82867,NaN,NaN,NaN,107633.00
90,89.0,NaN,Шаповаленко И. В.,Заместитель главного бухг,173000.0,NaN,15.0,NaN,NaN,NaN,...,NaN,221834.77,27364.55,NaN,NaN,27364.55,NaN,NaN,NaN,194470.22
91,90.0,NaN,Шацев А. Г.,Программист 1С,240000.0,NaN,22.0,NaN,NaN,NaN,...,NaN,60000.00,26100,NaN,NaN,26100,NaN,NaN,NaN,33900.00


In [54]:
# Список столбцов, которые мы оставляем (измерения)
id_vars = ['Сотрудник сокр', 'Должность']

# Делаем "расплавление" таблицы
df_melted = df.melt(
    id_vars=id_vars, 
    var_name='Вид_начисления', 
    value_name='Начислено'
)
df_melted = df_melted[df_melted['Начислено'].notna() & (df_melted['Начислено'] != 0)]

In [55]:
df_melted

,Сотрудник сокр,Должность,Вид_начисления,Начислено
26,Карукес С.,Мобильный разработчик,Больничный за счет работодателя,2648.76
48,Морозов С. В.,Ведущий технический консу,Больничный за счет работодателя,17021.91
97,Винниченко А. В.,Менеджер по развитию бизн,Командировка,35769.78
150,Ролдугин Д. В.,Директор по развитию зару,Командировка,62407.86
260,Ткач Д. Г.,Экономист по планированию,Компенсация не использованных дней отдыха,6521.74
...,...,...,...,...
727,Яковлев Т. И.,Тестировщик,Отпуск основной,89626.74
737,Гаджиева А. Л.,Младший веб-разработчик,Районный коэффициент,38500.00
828,Гаджиева А. Л.,Младший веб-разработчик,Северная надбавка,10150.00
916,Винниченко А. В.,Менеджер по развитию бизн,Суточные сверх нормы,5800.00


### Функции для обработки:

In [59]:
%load_ext autoreload
%autoreload 2

from python_files.handlers import ZupReportProcessor as ZUPpr
from python_files.handlers import T51

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import os
print(os.getcwd())  # Должен напечатать путь к вашей папке с проектом
print(os.listdir()) # Проверьте, есть ли в списке 'handlers.py'

d:\Основные данные\Задачи\ts21\Динамическая Т-51 с видами начислений\Источники данных для проверки
['.vscode', 'env', 'python_files', 'Данные ЗУП.xlsx', 'Данные Т51.xlsx', 'Запросы для проверки точности.sel']


In [60]:
proc = ZUPpr('Данные ЗУП.xlsx')
final_df = proc.run()

In [61]:
final_df

,Период,Сотрудник,Подразделение,Должность,Вид_начисления,Начислено
0,2025-09-01,Кова Евгений Романович,Финансовый отдел,Юрист,Оплата по окладу,115000.00
1,2025-09-01,Костюк Анастасия Владиславовна,Финансовый отдел,Юрисконсульт,Оплата по окладу,80500.00
2,2025-09-01,Самойлова Юлия Владимировна,Финансовый отдел,Бухгалтер,Оплата по окладу,138000.00
3,2025-09-01,Серая Марина Анатольевна,Финансовый отдел,Главный бухгалтер,Оплата по окладу,169272.73
4,2025-09-01,Серая Марина Анатольевна,Финансовый отдел,Главный бухгалтер,Отпуск основной,225024.75
...,...,...,...,...,...,...
811,2026-02-01,Расцветаев Николай Вадимович,"ОП Краснодар Березанская ООО ""Винтео""",Младший программист клиент-серверных приложений,Больничный за счет работодателя,5926.38
812,2026-02-01,Сейлер Константин Сергеевич,"ОП Краснодар Березанская ООО ""Винтео""",Инженер по автоматизации тестирования,Оплата по окладу,138000.00
813,2026-02-01,Сейлер Константин Сергеевич,"ОП Краснодар Березанская ООО ""Винтео""",Инженер по автоматизации тестирования,Больничный за счет работодателя,10962.27
814,2026-02-01,Чеботарев Алексей Евгеньевич,"ОП Краснодар Березанская ООО ""Винтео""",Старший веб-программист,Оплата по окладу,317000.00


In [62]:
proc = T51('Данные Т51.xlsx')
final_df_T51 = proc.run()

In [63]:
final_df_T51

,Сотрудник сокр,Должность,Вид_начисления,Начислено,Период
0,Карукес С.,Мобильный разработчик,Больничный за счет работодателя,2648.76,2025-09-01
1,Морозов С. В.,Ведущий технический консу,Больничный за счет работодателя,17021.91,2025-09-01
2,Винниченко А. В.,Менеджер по развитию бизн,Командировка,35769.78,2025-09-01
3,Ролдугин Д. В.,Директор по развитию зару,Командировка,62407.86,2025-09-01
4,Ткач Д. Г.,Экономист по планированию,Компенсация не использованных дней отдыха,6521.74,2025-09-01
...,...,...,...,...,...
859,Яковлев Т. И.,Тестировщик,Отпуск основной,89626.74,2026-02-01
860,Гаджиева А. Л.,Младший веб-разработчик,Районный коэффициент,38500.00,2026-02-01
861,Гаджиева А. Л.,Младший веб-разработчик,Северная надбавка,10150.00,2026-02-01
862,Винниченко А. В.,Менеджер по развитию бизн,Суточные сверх нормы,5800.00,2026-02-01
